In [1]:
import ccxt
import pandas as pd
import numpy as np
from scipy.signal import argrelextrema
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
import datetime
bybit = ccxt.bybit()
from freqtrade_client import FtRestClient
from datetime import datetime, date, timedelta, timezone, time
import pandas as pd
import os
server_url = 'http://127.0.0.1:8080'
username = ''
password = ""
client = FtRestClient(server_url, username, password)

In [21]:
trades = pd.DataFrame(client.trades().get('trades'))

starting_balance = client.daily(1).get('data')[0].get('starting_balance')
today_loss = trades[
    (trades.close_profit_abs < 0) & 
    (trades.close_date > date.today().strftime('%Y-%m-%d'))
].close_profit_abs.sum().item() / starting_balance

week_day = date.weekday(date.today())
open_date = (date.today() - timedelta(days=week_day))
starting_balance = client.weekly(1).get('data')[0].get('starting_balance')
this_week_loss = trades[
    (trades.close_profit_abs < 0) & 
    (trades.close_date > open_date.strftime('%Y-%m-%d'))
].close_profit_abs.sum().item() / starting_balance

In [ ]:
today_loss

In [ ]:
today_loss

In [ ]:
week_day = date.weekday(date.today())
open_date = (date.today() - timedelta(days=week_day))
open_date

In [ ]:
all_trades = client.trades().get('trades') + client.status()
dataframe = pd.DataFrame(all_trades)
dataframe.is_open.any()

In [3]:
def return_order_book(symbol='BTC/USDT', n=200):
    bybit_ob = bybit.fetchOrderBook(symbol, n)
    binance_ob = bybit.fetchOrderBook(symbol, n)
    kucoin_ob = bybit.fetchOrderBook(symbol, n)
    bid_values = {
        'price': np.hstack((np.array(bybit_ob['bids'])[:,0], np.array(binance_ob['bids'])[:,0], np.array(kucoin_ob['bids'])[:,0])),
        'volume': np.hstack((np.array(bybit_ob['bids'])[:,1], np.array(binance_ob['bids'])[:,1], np.array(kucoin_ob['bids'])[:,1])),
        'side':'bid'
    }
    ask_values = {
        'price': np.hstack((np.array(bybit_ob['asks'])[:,0], np.array(binance_ob['asks'])[:,0], np.array(kucoin_ob['asks'])[:,0])),
        'volume': np.hstack((np.array(bybit_ob['asks'])[:,1], np.array(binance_ob['asks'])[:,1], np.array(kucoin_ob['asks'])[:,1])),
        'side':'ask'
    }
    bid_dataframe = pd.DataFrame(bid_values)
    ask_dataframe = pd.DataFrame(ask_values)
    dataframe = pd.concat((bid_dataframe,ask_dataframe))
    dataframe = dataframe.groupby(['price','side']).sum().reset_index()
    # dataframe.groupby('side').sum()
    dataframe['now'] = datetime.datetime.now()
    return dataframe

In [3]:
def plot(dataframe, trades=[]):

    fig = go.Figure(data=[go.Candlestick(x=dataframe.date.values,
                    open=dataframe['open'],
                    high=dataframe['high'],
                    low=dataframe['low'],
                    close=dataframe['close'],
                    increasing_line_color= 'green', 
                    decreasing_line_color= 'red')])

    fig.add_scatter(
        x= dataframe.date.values, 
        y= dataframe.upper_band.values,
        mode="lines", 
        marker=dict(size=7, color="green")
    )

    fig.add_scatter(
        x= dataframe.date.values, 
        y= dataframe.y.values,
        mode="lines", 
        marker=dict(size=7, color="blue")
    )

    fig.add_scatter(
        x= dataframe.date.values, 
        y= dataframe.lower_band.values,
        mode="lines", 
        marker=dict(size=7, color="red")
    )

    fig.add_scatter(
        x= dataframe[dataframe.extrema == 1].date.values, 
        y= dataframe[dataframe.extrema == 1].high.values,
        mode="markers", 
        marker=dict(size=7, color="purple")
    )

    fig.add_scatter(
        x= dataframe[dataframe.extrema == -1].date.values, 
        y= dataframe[dataframe.extrema == -1].low.values,
        mode="markers", 
        marker=dict(size=7, color="yellow")
    )

    if not trades.empty:
        fig.add_scatter(
            x= trades.open_date.values, 
            y= trades.open_rate.values,
            mode="markers", 
            marker=dict(size=15, color="green")
        )

        fig.add_scatter(
            x= trades.close_date.values, 
            y= trades.close_rate.values,
            mode="markers", 
            marker=dict(size=15, color="red")
        )
    fig.update_xaxes(showgrid=False)
    fig.update_yaxes(showgrid=False)
    fig.update_layout(autosize=True, height=500,xaxis_rangeslider_visible=False)
    fig.show()

In [4]:
def calculate_extrema(dataframe, kernel=6):
    dataframe["extrema"] = 0
    min_peaks = argrelextrema(dataframe["low"].values, np.less_equal, order=kernel)
    max_peaks = argrelextrema(dataframe["high"].values, np.greater_equal, order=kernel)
    for mp in min_peaks[0]:
        dataframe.at[mp, "extrema"] = -1
    for mp in max_peaks[0]:
        dataframe.at[mp, "extrema"] = 1
    dataframe['last_min_peak'] = dataframe.at[min_peaks[0][-1], "low"]
    dataframe['last_max_peak'] = dataframe.at[max_peaks[0][-1], "high"]
    dataframe['h_dist'] = np.where(dataframe.extrema == 1, (dataframe.high - dataframe.upper_band), 0)
    dataframe['l_dist'] = np.where(dataframe.extrema == -1, (dataframe.lower_band - dataframe.low), 0)
    dataframe['h_ratio'] = dataframe['h_dist'] / dataframe['band_dist']
    dataframe['l_ratio'] = dataframe['l_dist'] / dataframe['band_dist']
    dataframe['l_h_ratio'] = dataframe.at[max_peaks[0][-1], "h_ratio"]
    dataframe['l_l_ratio'] = dataframe.at[min_peaks[0][-1], "l_ratio"]
    dataframe['last_max'] = dataframe.at[max_peaks[0][-1], "close"]
    dataframe['last_min'] = dataframe.at[min_peaks[0][-1], "close"]
    return dataframe

In [5]:
def caculate_regression(dataframe, kernel=1440):
    dataframe_ = dataframe.copy()[-kernel:]
    x = dataframe_.index.values.reshape(-1, 1)
    y = dataframe_.close.values
    model = LinearRegression()
    model.fit(x, y)
    x = dataframe.index.values.reshape(-1, 1)
    dataframe['y'] = model.predict(x)
    dataframe['coef'] = float(model.coef_[0])
    dataframe['upper_band'] = dataframe['y'] + dataframe.high.std()
    dataframe['lower_band'] = dataframe['y'] - dataframe.low.std()
    dataframe['band_dist'] = dataframe['upper_band'] - dataframe['lower_band']
    return dataframe

In [ ]:
!docker-compose run --rm TradeStrategy download-data -c user_data/config.json --timeframe 1m

In [13]:
def return_dataframe_from_csv(pair, columns=[]):
    dataframe = pd.read_csv(f'df_{pair}.csv')
    dataframe['date'] = pd.to_datetime(dataframe['date'])
    if columns:
        dataframe = dataframe[columns]
    return dataframe

In [30]:
files = [file for file in os.listdir(".") if file.endswith('.csv')]
tickers = [file[3:-4] for file in files]

In [36]:
def plot_ticker(ticker):
# dataframe = pd.read_feather("../data/bybit/futures/BTC_USDT_USDT-1m-futures.feather")
    columns=['date','open','high','low','close','volume']
    dataframe = return_dataframe_from_csv(ticker)
    # open_time = '2024-11-19 12:42'
    # close_time = '2024-11-20 12:42'
    now = datetime.datetime.now()
    close_time = now.strftime("%Y-%m-%d %H:%M:%S")
    open_time = (now - datetime.timedelta(days=1)).strftime("%Y-%m-%d %H:%M:%S")
    # dataframe = dataframe[(dataframe.date > open_time) & (dataframe.date <= close_time)].reset_index()
    # dataframe = caculate_regression(dataframe, kernel=1440)
    # dataframe = calculate_extrema(dataframe, kernel=6)
    trades = pd.DataFrame(client.trades().get('trades') + client.status())
    trades = trades[(trades.open_date > open_time) & (trades.close_date <= close_time)].reset_index()
    last_candle = dataframe.iloc[-1].squeeze()
    print(float((last_candle['upper_band'] - last_candle['y']) / (last_candle['y'] - last_candle['lower_band'])))
    print("Coef:", dataframe.iloc[-1].squeeze()['coef'])
    print(ticker)
    plot(dataframe,trades=pd.DataFrame())

In [ ]:
plot_ticker(tickers[7])

In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [3]:
dataframe = pd.read_csv("AVAX_8_df.csv")
# trades = pd.read_csv(f'trades.csv')

In [ ]:
print("Down: ", dataframe['down'].iloc[-240:].count())
print("Up: ", dataframe['up'].iloc[-240:].count())

In [ ]:
dataframe[["&s-up_or_down","down","up","enter_long", "enter_short", "do_predict"]].iloc[-12:]["&s-up_or_down"]

In [ ]:
import os

tickers = trades.pair.to_dict()
dfs = [f"{value[:-10]}_{key + 1}_df.csv" for  key, value in tickers.items()]
obs = [f"{value[:-10]}_{key + 1}_ob.csv" for  key, value in tickers.items()]
files = sorted(dfs + obs)
files.append(["analysis.ipynb", "trades.csv"])
for f in os.listdir('.'):
    if f not in files:
        os.unlink(f)

In [11]:
def caculate_coef(window):
    x = np.arange(len(window)).reshape(-1, 1)
    y = window
    model = LinearRegression()
    model.fit(x, y)
    x = dataframe.index.values.reshape(-1, 1)
    return model.coef_[0]

def calculate_coef_window(dataframe, window):
    dataframe['coef'] = dataframe.close.rolling(window=window).apply(caculate_coef)
    return dataframe


In [ ]:
fig = make_subplots(rows=3, cols=1)

dataframe = pd.read_csv("XRP_2_df.csv")
dataframe = dataframe[['date','open','high','low','close','volume']]

dataframe = calculate_coef_window(dataframe, window=12)

fig.add_trace(
    go.Scatter(
        x= dataframe.date.values, 
        y= dataframe.close.values,
        mode="lines", 
        marker=dict(size=7, color="green")
    ),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(
        x= dataframe.date.values, 
        y= dataframe.coef.values,
        mode="lines", 
        marker=dict(size=7, color="blue")
    ),
    row=3, col=1
)

fig.show()

In [1]:
import os
import re
import pandas as pd

In [17]:
# def correlated_ticker(ticker):
ticker = 'AVAX'
path = "../data/bybit/futures"
all_files = os.listdir(path)
files = [file for file in all_files if '_USDT_USDT-15m-futures' in file]
tickers = {file[:-30]:pd.read_feather(path + "/" + file).close for file in files}
dataframe = pd.DataFrame(tickers).ffill()
corr_df = dataframe.corr()
max_corrs = corr_df[(corr_df != 1)].max()
corr_df[max_corrs == max_corrs.max()].index.to_list()


['XRP', 'HBAR']

In [16]:
corr_df = dataframe.corr()
max_corrs = corr_df[(corr_df != 1)].max()
corr_df[max_corrs == max_corrs.max()].index.to_list()

['XRP', 'HBAR']

In [18]:
dataframe

,GOAT,ETH,XLM,SOL,FTM,ME,ONDO,PNUT,SHIB1000,1000PEPE,...,AVAX,DOGE,BTC,ADA,LINK,SUI,HBAR,WIF,1000000BABYDOGE,MOVE
0,0.7823,2967.20,0.10102,200.53,0.7215,3.0800,0.7376,0.4180,0.018848,0.010822,...,28.410,0.19746,76155.2,0.4432,13.486,2.2648,0.05270,2.3150,0.002334,0.7010
1,0.7841,2966.18,0.10071,200.28,0.7211,2.6873,0.7337,0.4281,0.018866,0.010786,...,28.450,0.19680,75974.4,0.4425,13.526,2.2646,0.05243,2.3143,0.002333,0.6595
2,0.7766,2951.67,0.10059,199.65,0.7173,2.6026,0.7324,0.4221,0.018921,0.010810,...,28.340,0.19727,76014.2,0.4438,13.517,2.2654,0.05234,2.3114,0.002325,0.6728
3,0.7794,2955.92,0.10041,199.64,0.7138,2.6910,0.7336,0.3966,0.018854,0.010768,...,28.375,0.19599,76035.0,0.4433,13.635,2.2588,0.05213,2.3024,0.002310,0.6199
4,0.7730,2935.89,0.10014,198.54,0.7075,2.6700,0.7259,0.3834,0.018745,0.010689,...,28.320,0.19573,76108.5,0.4402,13.499,2.2487,0.05169,2.2835,0.002297,0.6136
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3248,0.7526,3910.28,0.43303,231.83,1.2680,4.7611,1.8123,1.2697,0.028866,0.024396,...,52.790,0.41486,99465.5,1.1349,28.195,4.6604,0.29552,3.0686,0.005123,0.6851
3249,0.7429,3910.28,0.43307,231.94,1.2660,4.7611,1.7968,1.2697,0.028827,0.024286,...,52.570,0.41540,99465.5,1.1344,28.418,4.5514,0.29504,3.0652,0.005125,0.6851
3250,0.7475,3910.28,0.43423,231.34,1.2706,4.7611,1.8049,1.2697,0.028872,0.024324,...,52.310,0.41513,99465.5,1.1386,28.239,4.5790,0.29566,3.0780,0.005112,0.6851
3251,0.7609,3910.28,0.44434,232.67,1.2728,4.7611,1.8217,1.2697,0.029199,0.024829,...,52.845,0.41776,99465.5,1.1515,28.282,4.6628,0.29997,3.1237,0.005115,0.6851


In [103]:
path = "../data/bybit/futures"
all_files = os.listdir(path)
files = [file for file in all_files if '_USDT_USDT-15m-futures' in file]
tickers = {file[:-30]:pd.read_feather(path + "/" + file).close for file in files}
dataframe = pd.DataFrame(tickers).ffill()

In [50]:
dataframe = pd.read_csv('corr_df.csv')

In [51]:
dataframe.corr()

,ETH/USDT:USDT,XRP/USDT:USDT,SOL/USDT:USDT,DOGE/USDT:USDT,LINK/USDT:USDT,1000PEPE/USDT:USDT,SUI/USDT:USDT,AAVE/USDT:USDT,ADA/USDT:USDT,AVAX/USDT:USDT,WIF/USDT:USDT,HBAR/USDT:USDT,ONDO/USDT:USDT,CRV/USDT:USDT,COW/USDT:USDT,ENS/USDT:USDT,GOAT/USDT:USDT,VIRTUAL/USDT:USDT,PNUT/USDT:USDT,LTC/USDT:USDT
ETH/USDT:USDT,1.000000,0.187678,0.672399,0.746033,0.555521,0.370180,0.704415,0.431396,0.440570,0.579590,0.642719,0.286621,0.279871,0.781887,0.610426,0.585233,0.622437,0.441433,0.763190,0.556142
XRP/USDT:USDT,0.187678,1.000000,0.570666,0.484604,0.362246,-0.052398,0.065614,-0.106168,0.746394,0.620030,0.583930,0.802030,0.307259,-0.127973,0.140436,0.535751,0.540908,-0.055336,0.538675,0.564136
SOL/USDT:USDT,0.672399,0.570666,1.000000,0.779998,0.324090,-0.144662,0.356911,-0.061508,0.850926,0.801046,0.791198,0.512071,0.357372,0.290084,0.281078,0.769366,0.890241,0.189156,0.882528,0.891162
DOGE/USDT:USDT,0.746033,0.484604,0.779998,1.000000,0.217674,0.276471,0.275550,-0.150902,0.668569,0.576132,0.930008,0.539788,-0.042597,0.404352,0.163074,0.484842,0.610663,-0.082607,0.900873,0.774728
LINK/USDT:USDT,0.555521,0.362246,0.324090,0.217674,1.000000,0.233125,0.756399,0.777037,0.355319,0.618474,0.148271,0.334445,0.612610,0.381255,0.858159,0.622076,0.504186,0.692845,0.354153,0.247783
1000PEPE/USDT:USDT,0.370180,-0.052398,-0.144662,0.276471,0.233125,1.000000,0.379874,0.400532,-0.354084,-0.214262,0.179456,-0.090697,-0.186580,0.403519,0.207814,-0.271097,-0.222138,0.070970,0.118656,-0.301408
SUI/USDT:USDT,0.704415,0.065614,0.356911,0.275550,0.756399,0.379874,1.000000,0.809961,0.160450,0.382739,0.180560,0.096615,0.496162,0.569474,0.814366,0.436109,0.467935,0.778827,0.459424,0.209563
AAVE/USDT:USDT,0.431396,-0.106168,-0.061508,-0.150902,0.777037,0.400532,0.809961,1.000000,-0.188642,0.146563,-0.246553,-0.111476,0.551184,0.483647,0.834625,0.229516,0.118310,0.806314,-0.012974,-0.239904
ADA/USDT:USDT,0.440570,0.746394,0.850926,0.668569,0.355319,-0.354084,0.160450,-0.188642,1.000000,0.863148,0.702083,0.712782,0.342932,0.023463,0.195919,0.813657,0.843939,0.022966,0.761159,0.920288
AVAX/USDT:USDT,0.579590,0.620030,0.801046,0.576132,0.618474,-0.214262,0.382739,0.146563,0.863148,1.000000,0.576470,0.519364,0.529575,0.258901,0.528890,0.936993,0.881088,0.343822,0.714078,0.806789


In [19]:
dataframe = pd.read_csv('AVAX_df.csv')

In [24]:
pairs = ['AVA', 'XRP']

In [21]:
dataframe['pair'] = "AVAX"

In [25]:
dataframe.pair.isin(pairs)

0       False
1       False
2       False
3       False
4       False
        ...  
1070    False
1071    False
1072    False
1073    False
1074    False
Name: pair, Length: 1075, dtype: bool